In [1]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os


load_dotenv()
conn = psycopg2.connect(
    host=os.getenv('DB_HOST'), port=os.getenv('DB_PORT'),
    dbname=os.getenv('DB_NAME'), user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD')
)


# Overview query
df_overview = pd.read_sql('''
    SELECT EXTRACT(YEAR FROM seance) as year,
           COUNT(*) as rows,
           COUNT(DISTINCT isin_code) as stocks,
           MIN(seance) as first_date,
           MAX(seance) as last_date
    FROM daily_prices
    GROUP BY year ORDER BY year
''', conn)
print(df_overview.to_string(index=False))


  year  rows  stocks first_date  last_date
2016.0 12719      51 2016-01-04 2016-12-30
2017.0 13305      53 2017-01-02 2017-12-29
2018.0 12768      52 2018-01-02 2018-12-31
2019.0 12948      52 2019-01-02 2019-12-31
2020.0 12394      51 2020-01-02 2020-12-31
2021.0 12710      57 2021-01-04 2021-12-31
2022.0 11262      45 2022-01-03 2022-12-30
2023.0 10044      41 2023-01-02 2023-12-29
2024.0 10542      42 2024-01-02 2024-12-31
2025.0 11703      47 2025-01-02 2025-12-31


C:\Users\Negza\AppData\Local\Temp\ipykernel_12444\1608230169.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_overview = pd.read_sql('''


In [2]:
df_companies = pd.read_sql('''
    SELECT ticker, isin_code, first_seen_date, last_seen_date, total_trading_days
    FROM company_metadata
    ORDER BY total_trading_days DESC
''', conn)
print(f'Total companies: {len(df_companies)}')
print(df_companies.to_string(index=False))


Total companies: 121
            ticker    isin_code first_seen_date last_seen_date  total_trading_days
               BNA TN0003100609      2021-01-04     2025-12-31                1257
ATELIER MEUBLE INT TN0007740012      2021-01-04     2025-12-31                1257
               UIB TN0003900107      2021-01-04     2025-12-31                1257
               ATB TN0003600350      2021-01-04     2025-12-31                1257
               TPR TN0007270010      2021-01-04     2025-12-31                1257
         AMEN BANK TN0003400058      2021-01-04     2025-12-31                1257
             ARTES TN0007300015      2021-01-04     2025-12-31                1257
          SOTRAPIL TN0006660013      2021-01-04     2025-12-31                1257
   CARTHAGE CEMENT TN0007400013      2021-01-04     2025-12-31                1257
ENNAKL AUTOMOBILES TN0007410012      2021-01-04     2025-12-31                1257
    TELNET HOLDING TN0007440019      2021-01-04     2025-12-31    

C:\Users\Negza\AppData\Local\Temp\ipykernel_12444\3442434290.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_companies = pd.read_sql('''


In [3]:
df_amen = pd.read_sql('''
    SELECT seance, cloture, ouverture, plus_haut, plus_bas,
           quantite_negociee, capitaux
    FROM daily_prices
    WHERE ticker = 'AMEN BANK'
    ORDER BY seance
''', conn)


df_amen['seance'] = pd.to_datetime(df_amen['seance'])
df_amen.set_index('seance', inplace=True)
print(f'AMEN BANK: {len(df_amen)} trading days')
print(df_amen.head(10))


AMEN BANK: 2508 trading days
            cloture  ouverture  plus_haut  plus_bas  quantite_negociee  \
seance                                                                   
2016-01-04    23.40      23.40      23.50     23.40                181   
2016-01-05    23.30      23.50      23.50     23.21                491   
2016-01-06    23.50      23.30      23.50     23.30               2631   
2016-01-07    23.45      23.50      23.70     23.10               1106   
2016-01-08    23.50      23.01      23.50     23.00               5053   
2016-01-11    23.50      23.50      23.50     23.23               1321   
2016-01-12    23.00      23.50      23.50     22.90               3049   
2016-01-13    23.40      23.40      23.40     23.40                100   
2016-01-15    22.90      23.40      23.40     22.90               2092   
2016-01-18    22.65      23.35      23.35     22.65               2708   

             capitaux  
seance                 
2016-01-04    4238.30  
2016-01-05

C:\Users\Negza\AppData\Local\Temp\ipykernel_12444\3742126083.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_amen = pd.read_sql('''


In [4]:
df_stats = pd.read_sql('''
    SELECT ticker,
           ROUND(AVG(cloture)::numeric, 3) as avg_close,
           ROUND(MIN(cloture)::numeric, 3) as min_close,
           ROUND(MAX(cloture)::numeric, 3) as max_close,
           ROUND(AVG(quantite_negociee)::numeric, 0) as avg_volume,
           COUNT(*) as trading_days
    FROM daily_prices
    WHERE cloture > 0
    GROUP BY ticker
    ORDER BY avg_volume DESC
    LIMIT 20
''', conn)
print(df_stats.to_string(index=False))


            ticker  avg_close  min_close  max_close  avg_volume  trading_days
   CARTHAGE CEMENT      1.856       1.28       2.31    126299.0          1256
TAWASOL GP HOLDING      0.699       0.52       0.95     54423.0           380
           SOMOCER      1.075       0.67       2.25     47014.0          2007
          TUNISAIR      0.601       0.39       0.93     40960.0          1363
              SFBT     17.694      11.10      27.20     39497.0          2507
               SAH     11.460       7.70      17.14     34243.0          2505
        SOTIPAPIER      4.985       2.51       7.90     29886.0          2480
    DELICE HOLDING     14.392      10.12      28.60     29485.0          2453
                BT      6.729       4.88      10.90     27726.0          2504
             ADWYA      5.128       2.14       9.12     27558.0          1748
              UADH      2.406       0.52       7.07     27262.0          1315
  ONE TECH HOLDING     10.657       6.60      18.78     25938.0 

C:\Users\Negza\AppData\Local\Temp\ipykernel_12444\2171677229.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_stats = pd.read_sql('''
